In [154]:
import json
import requests
import pandas as pd
import polars as pl

from pathlib import Path
from typing import List, Dict, Any
from tqdm import tqdm

from bs4 import BeautifulSoup


In [141]:

THIS_FOLDER = Path('.')

ABT_FOLDER = THIS_FOLDER / 'abt'
RAW_FOLDER = THIS_FOLDER / 'raw'


In [130]:

def _get_html_soup(url: str) -> BeautifulSoup:
    
    void_soup = BeautifulSoup("", "html.parser")

    try:
        response = requests.get(url)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch {url}: {response.status_code}")
    
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return void_soup
    
    try:
        return BeautifulSoup(response.text, "html.parser")
    
    except Exception as e:
        print(f"Error parsing HTML from {url}: {e}")
        return void_soup


In [131]:

def _get_most_accessed_list(genre: str, soup: BeautifulSoup, amount: int) -> List[Dict[str, str]]:
    
    i, songs = 0, []

    try:
        ol = soup.find("ol", class_="top-list_mus --top")
        ol_as = ol.find_all("a")
    
    except Exception as e:
        print(f"Error extracting most accessed list: {e}")
        return songs

    for a in ol_as:
        
        i += 1
        if i > amount:
            break

        try:
            href = a.get("href")
            title = a.find("b", class_="font --base --strong --size16").text
            artist = a.find("span", class_="font --base --size14").text
            song = {
                "genre": genre,
                "index": i,
                "title": title,
                "artist": artist,
                "link": href
            }

            songs.append(song)

        except Exception as e:
            print(f"Error extracting song information: {e} from {a}")
            continue

    return songs



In [132]:

def scrape_top_songs(genre: str, amount: int = 1000) -> List[Dict[str, str]]:
    
    base = "https://www.letras.mus.br/"
    
    url = f"{base}/mais-acessadas/{genre}/"
    soup = _get_html_soup(url)
    
    songs = _get_most_accessed_list(genre, soup, amount)
    return songs


In [133]:

def _get_lyrics(soup_song: BeautifulSoup) -> List[str]:
    
    paragraphs = []

    try:
        div_lyric = soup_song.find("div", class_="lyric")
        div_lyric_original = div_lyric.find("div", class_="lyric-original")
        div_paragraphs = div_lyric_original.find_all("p")
    except AttributeError as e:
        print(f"Could not find lyrics in the page. Error {e}")
        return paragraphs

    try:

        for p in div_paragraphs:
            fixed_texts = [
                str(p_content).strip().replace("<br/>", "\n")
                for p_content in p.contents
            ]

            text = "".join(fixed_texts)
            paragraphs.append(text)
            
    except Exception as e:
        print(f"Error while processing lyrics: {e}")

    return paragraphs

In [134]:

def _get_n_views(soup_song: BeautifulSoup) -> int:
    n_views = 0

    try:    
        div_title = soup_song.find("div", class_="head --lyric")
        div_head_info = div_title.find("div", class_="head-info")
        div_head_info_container = div_head_info.find("div", class_="head-info-container")
        div_head_info_container_b = div_head_info_container.find("b")
    
    except AttributeError as e:
        print(f"Could not find the number of views for this song. Error: {e}")
        return n_views

    try:
        str_n_views = str(div_head_info_container_b.text)
        n_views = int(str_n_views.split()[0].replace(".", ""))
    
    except (AttributeError, ValueError) as e:
        print(f"Could not parse the number of views for this song. Error: {e}")
        return n_views
    
    return n_views


In [135]:

def get_song_data(song_link: str) -> Dict[str, Any]:

    base = "https://www.letras.mus.br/"
    url_song = f"{base}{song_link}/"

    soup_song = _get_html_soup(url=url_song)
    
    lyrics = _get_lyrics(soup_song)
    n_views = _get_n_views(soup_song)

    song_data = {
        "lyrics": lyrics,
        "n_views": n_views
    }

    return song_data


In [142]:

genres = RAW_FOLDER / "letras_mus_br" / "genres.json"

if not genres.exists():
    raise FileNotFoundError(f"File {genres} not found. Please run the script to scrape the genres first.")

with open(genres, "r") as f:
    genres = json.load(f)

musical_genres = genres.get("musical_genres", [])
print("Amount of genres:", len(musical_genres))


Amount of genres: 94


In [156]:

def get_df_songs(genre: str) -> pl.DataFrame:
        
    songs = scrape_top_songs(genre, amount=1000)

    df_songs = pl.DataFrame(songs)
    df_songs = (

        
        df_songs
        
        # Retrieve song data (lyrics and number of views) for each song link
        .with_columns(
            pl.col("link").map_elements(
                get_song_data,
                return_dtype=pl.Struct([pl.Field("lyrics", pl.List(pl.Utf8)), pl.Field("n_views", pl.Int64)])
            ).alias("data")
        )


        # Extract json to separate columns for number of views and lyrics
        .with_columns(
            pl.col("data").map_elements(lambda x: x.get("n_views", 0)).alias("n_views"),
            pl.col("data").map_elements(lambda x: x.get("lyrics", [])).alias("lyrics")
        )

        # Drop the intermediate "data" column
        .drop("data")

        # Explode the lyrics list to have one row per paragraph
        .explode("lyrics")

        # Add a paragraph number column to keep track of the order of paragraphs
        .with_columns(
            pl.col("lyrics").cum_count().over("index").alias("paragraph")
        )

    )

    return df_songs


In [164]:

def check_if_already_scraped(filepath: Path, genre: str) -> bool:
    already_existing_genres = pl.read_parquet(filepath).select(pl.col("genre").unique()).to_series().to_list()
    return genre in already_existing_genres

def get_filepath(genre: str = None) -> Path:
    
    if not ABT_FOLDER.exists():
        ABT_FOLDER.mkdir(parents=True)

    return ABT_FOLDER / f"3_abt_song_lyric_views{'_' + genre if genre else ''}.parquet"


In [159]:

def save_abt(filepath: Path, df_songs: pl.DataFrame, genre: str) -> None:

    if filepath.exists():

        # Delete genre if already exists in the file to avoid duplicates
        already_scrapped = check_if_already_scraped(filepath, genre)
        if already_scrapped:
            print(f"Genre `{genre}` already exists in the file. Deleting existing entries to avoid duplicates.")
            df_existing = pl.read_parquet(filepath)
            df_existing = df_existing.filter(pl.col("genre") != genre)
            df_existing.write_parquet(filepath)

        # Append new data to the file
        print(f"Appending `{genre}` to existing parquet.")
        existing_df = pl.read_parquet(filepath)
        combined_df = existing_df.vstack(df_songs)
        combined_df.write_parquet(filepath)

    if not filepath.exists():
        print(f"Writing new parquet with `{genre}`.")
        df_songs.write_parquet(filepath)


In [166]:

filepath = get_filepath()

for genre in tqdm(musical_genres, desc="Processing genres"):
    already_scrapped = check_if_already_scraped(filepath, genre)

    if already_scrapped:
        print(f"Genre `{genre}` already exists in the file. Skipping to avoid duplicates.")
        continue
    
    genre_filepath = get_filepath(genre)
    if genre_filepath.exists():
        print(f"Genre `{genre}` already exists in the file. Skipping to avoid duplicates.")
        continue

    df_songs = get_df_songs(genre)
    save_abt(genre_filepath, df_songs, genre)
    


Processing genres:   4%|▍         | 4/94 [00:00<00:02, 39.17it/s]

Genre `afrobeats` already exists in the file. Skipping to avoid duplicates.
Genre `alternativo-indie` already exists in the file. Skipping to avoid duplicates.
Genre `arrocha` already exists in the file. Skipping to avoid duplicates.
Genre `axe` already exists in the file. Skipping to avoid duplicates.
Genre `bachata` already exists in the file. Skipping to avoid duplicates.
Genre `blues` already exists in the file. Skipping to avoid duplicates.
Genre `bolero` already exists in the file. Skipping to avoid duplicates.


Processing genres:   9%|▊         | 8/94 [07:30<1:34:57, 66.25s/it]

Writing new parquet with `bossa-nova`.


Processing genres:  10%|▉         | 9/94 [15:01<3:08:35, 133.12s/it]

...

Writing new parquet with `xote`.


Processing genres:  99%|█████████▉| 93/94 [10:37:30<07:10, 430.95s/it]

Writing new parquet with `zamba`.
Could not parse the number of views for this song. Error: 'NoneType' object has no attribute 'text'


Processing genres: 100%|██████████| 94/94 [10:44:44<00:00, 411.54s/it]

Writing new parquet with `zouk`.
